## 95. Tile Coverage
0. Packages
1. Settings
2. Check Coverage

### 0. Packages

In [1]:
# Packages
import glob
import geopandas as gpd
import os
from tqdm import tqdm
import rioxarray as rxr

### 2. Settings

In [3]:
# File paths tiles
file_path_tiles_parquet = r'p:\11209821-cmems-global-sdb\00_miscellaneous\AOI_polygons_world\df_boxes_world_Z10_filtered_v2.parquet'

file_path_result_tifs = glob.glob(r'p:\11209821-cmems-global-sdb\01_intertidal\02_data\05_calibrated\intertidal_improved_100m_global\05_reprojected\*.tif')

# Number of files
print(f'Number of result files: {len(file_path_result_tifs)}')

Number of result files: 11893


### Check coverage

In [ ]:
data_coverages = {}
for file_path_result_tif in tqdm(file_path_result_tifs, desc='Processing files'):
    # Open file
    da = rxr.open_rasterio(file_path_result_tif).squeeze()

    # Get name
    name = os.path.basename(file_path_result_tif).split('_t')[0]
    
    # Compute data coverage
    data_coverage = da.notnull().sum().item() / da.size * 100

    # Append data coverage
    data_coverages[name] = data_coverage

Processing files: 100%|██████████| 11515/11515 [26:00<00:00,  7.38it/s]


In [78]:
# Read tiles
gdf_tiles = gpd.read_parquet(file_path_tiles_parquet)
gdf_tiles['nearest_station_distance'] = gdf_tiles['nearest_station_distance']/1000  # Convert to km
gdf_tiles['intertidal_coverage'] = gdf_tiles['intertidal_coverage']*100  # Convert to percentage
gdf_tiles['intertidal_coverage_ed'] = gdf_tiles['intertidal_coverage_ed']*100  # Convert to percentage
gdf_tiles['processed'] = (gdf_tiles['nearest_station_distance'] < 37) & (gdf_tiles['intertidal_coverage'] > 0) & (gdf_tiles['intertidal_coverage_ed'] > 1)

# Add data coverage
gdf_tiles['data_coverage'] = gdf_tiles['name'].map(data_coverages)
gdf_tiles['data_coverage'] = gdf_tiles['data_coverage'].fillna(0)

In [79]:
# Save parquet
gdf_tiles.to_parquet(file_path_tiles_parquet.replace('_v2.parquet', '_v3.parquet'), index=False)

In [89]:
# Number of tiles
print(f'Number of tiles: {len(gdf_tiles[gdf_tiles['data_coverage'] > 2])}')

Number of tiles: 2673


In [88]:
gdf_tiles[gdf_tiles['data_coverage'] > 2].explore(column='data_coverage')